[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S2_hoja_derivadas.ipynb)

# Hoja: columnas para tu estación

**Al terminar podrás** escoger columnas derivadas para un pronóstico y decir, con un número, cuánto mejoran;
y reconocer un modelo con fuga de información aunque te lo presenten con un error muy bajo.

Sección 2 de S2, 15 minutos en sala. No se entrega ni se califica.

---

**De:** Operaciones MiBici<br>
**Asunto:** Pronóstico de salidas por hora en tu estación

Hoy pronosticamos las salidas de cada hora con una recta que solo usa dos rezagos: lo que salió ayer
a esa hora y lo que salió hace una semana a esa hora. Queremos mejorarlo para tu estación.
Te dejamos siete columnas candidatas. Escoge **dos** para agregar y dinos cuánto baja el error en julio,
qué tan seguro estás y qué recomiendas.

---

**Cómo trabajar.** Una persona comparte pantalla y escribe; las demás dictan. Corran las celdas en orden.
Solo se edita la celda del paso 2.

Corre las dos celdas de abajo hasta que aparezca `✅ entorno listo`.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from dammg_utils import COLORES

du.estilo_dammg()

## Paso 1. Los datos y las columnas candidatas (4 min)

Cada fila es una estación en una hora; `departures` son las bicis que salieron. Train va de octubre a junio y
julio es para medir. La celda de abajo crea las columnas candidatas que no vienen en los datos.

| Columna | Familia | Qué le dice al modelo |
|---|---|---|
| `hour_of_day` | calendario | la hora del día; entra como 24 columnas de 0 y 1 |
| `is_weekend` | calendario | 1 si es sábado o domingo |
| `pico` | calendario | 1 de 7 a 9 y de 17 a 19 |
| `quincena` | calendario | 1 si es día 15 o 30 del mes |
| `departures_mean_168h` | promedio móvil | qué tan movida anduvo la estación la última semana |
| `prom_est_hora` | por grupo | el promedio de esa estación a esa hora y tipo de día, **calculado solo con train** |
| `est_hora` | interacción | una columna de 0 y 1 por cada pareja estación y hora; el modelo aprende su propio peso |

In [ ]:
df = pd.read_parquet(RUTA_DATOS)

# Calendario: se derivan de la hora
df["pico"] = df.hour_of_day.isin([7, 8, 9, 17, 18, 19]).astype(int)
df["quincena"] = df.hour.dt.day.isin([15, 30]).astype(int)

# Por grupo: el promedio de cada estación, hora y tipo de día, con train y nada más
tabla = df[df.particion == "train"].groupby(["station_id", "hour_of_day", "is_weekend"]).departures.mean()
df = df.join(tabla.rename("prom_est_hora"), on=["station_id", "hour_of_day", "is_weekend"])

# Interacción: una etiqueta por cada pareja estación y hora (después se vuelve columnas de 0 y 1)
df["est_hora"] = df.station_id.astype(str) + "_" + df.hour_of_day.astype(str)

train = df[df.particion == "train"]   # octubre a junio: para ajustar
julio = df[df.particion == "val"]     # julio: para medir
print(f"train: {len(train):,} filas   julio: {len(julio):,} filas")

Así se ve el día típico de una estación, según train. Cambia el número si quieres ver otra; en el paso 2 escogen la suya.

In [ ]:
estacion_vista = 51   # cámbiala si quieres ver otra estación

perfil = train[train.station_id == estacion_vista].groupby(["hour_of_day", "is_weekend"]).departures.mean().reset_index()
perfil["tipo de día"] = perfil.is_weekend.map({0: "entre semana", 1: "fin de semana"})
ax = sns.lineplot(data=perfil, x="hour_of_day", y="departures", hue="tipo de día", marker="o",
                  palette=[COLORES["w"], COLORES["lambda"]])
ax.set(title=f"Estación {estacion_vista}: salidas promedio por hora (oct a jun)", xlabel="hora del día",
       ylabel="salidas por hora")
plt.show()

## Paso 2. El modelo (5 min)

Edita **solo** esta celda: tu estación y las dos columnas que quieres agregar (entre comillas, separadas por coma).
Las estaciones van de la 11 a la 333; algunas: 11, 35, 38, 50, 51, 52, 54, 194, 271.

Antes de correr, anoten aquí cuál creen que baja más el error.

In [ ]:
estacion = 51
mis_columnas = ["pico", "quincena"]

In [ ]:
# @title Las funciones del modelo: solo córrela
REZAGOS = ["departures_lag_24h", "departures_lag_168h"]
CATEGORIAS = ["hour_of_day", "est_hora"]     # estas se vuelven columnas de 0 y 1


def ajusta(columnas):
    """Recta ajustada con train (las 40 estaciones) con las columnas que le pases."""
    cat = [c for c in columnas if c in CATEGORIAS]
    num = [c for c in columnas if c not in CATEGORIAS]
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat), ("num", "passthrough", num)])
    return make_pipeline(prep, LinearRegression()).fit(train[columnas], train.departures)


def error_julio(modelo, columnas):
    """MAE en julio, solo en tu estación: cuántas bicis por hora se equivoca, en promedio."""
    mia = julio[julio.station_id == estacion]
    return mean_absolute_error(mia.departures, modelo.predict(mia[columnas]))


base = ajusta(REZAGOS)
nuevo = ajusta(REZAGOS + mis_columnas)
print("Listo: modelo de hoy y modelo con", mis_columnas)

## Paso 3. La medición (5 min)

La pregunta es "¿cuánto se equivoca por hora?", así que medimos con el **MAE** en julio, en bicis por hora y solo en tu estación.

In [ ]:
e_base = error_julio(base, REZAGOS)
e_nuevo = error_julio(nuevo, REZAGOS + mis_columnas)
print(f"Estación {estacion}, julio")
print(f"  modelo de hoy (rezagos):      {e_base:.2f} bicis por hora")
print(f"  con {mis_columnas}: {e_nuevo:.2f} bicis por hora")
print(f"  cambio: {e_nuevo - e_base:+.2f} bicis ({(e_nuevo - e_base) / e_base:+.0%})")

¿Hubo una mejor pareja? La celda de abajo prueba cada candidata **sola** junto a los rezagos (tarda unos 30 segundos).
Si encuentran algo mejor, regresen al paso 2, cambien la lista y vuelvan a correr.

In [ ]:
candidatas = ["hour_of_day", "is_weekend", "pico", "quincena", "departures_mean_168h", "prom_est_hora", "est_hora"]
resultados = {c: error_julio(ajusta(REZAGOS + [c]), REZAGOS + [c]) for c in candidatas}
tabla_candidatas = pd.Series(resultados).sort_values().round(2)
print(f"Estación {estacion}. Modelo de hoy: {e_base:.2f}. Con cada candidata, sola:")
tabla_candidatas

**Si su estación es la 35:** no tuvo salidas del 12 de junio al 20 de julio. Las columnas por grupo y de interacción
recuerdan una estación abierta y empeoran; los rezagos y el promedio de la semana sí se enteran del cierre.

Así se ve una semana de julio en tu estación con los dos modelos:

In [ ]:
semana = julio[(julio.station_id == estacion) & (julio.hour >= "2026-07-13") & (julio.hour < "2026-07-20")]
grafica = pd.DataFrame({"lo que salió": semana.departures.values,
                        "modelo de hoy": base.predict(semana[REZAGOS]),
                        "con tus columnas": nuevo.predict(semana[REZAGOS + mis_columnas])},
                       index=semana.hour.values)
ax = sns.lineplot(data=grafica, dashes=False, palette=[COLORES["tinta"], COLORES["error"], COLORES["w"]])
ax.set(title=f"Estación {estacion}, semana del 13 de julio", xlabel="", ylabel="salidas por hora")
plt.xticks(rotation=30)
plt.show()

## Paso 4. La respuesta (3 min)

Contesten a Operaciones en tres renglones, en esta celda (doble clic para editar):

1. **El número:** Con ___ y ___, el error en julio en la estación ___ pasa de ___ a ___ bicis por hora (___ %).
2. **Qué tan seguro:** Lo medí en un solo mes, julio, con ___ horas. (¿Julio fue un mes normal en tu estación?)
3. **Recomendación y riesgo:** Recomiendo ___. Riesgo: ___ (por ejemplo, si la estación cambia de patrón,
   el promedio de train no se entera).

In [ ]:
# Para copiar el renglón 1 sin errores de dedo
print(f"Con {' y '.join(mis_columnas)}, el error en julio en la estación {estacion} pasa de "
      f"{e_base:.2f} a {e_nuevo:.2f} bicis por hora ({(e_nuevo - e_base) / e_base:+.0%}).")
print(f"Horas de julio medidas: {(julio.station_id == estacion).sum()}")

---

**De:** Operaciones MiBici<br>
**Asunto:** RE: Pronóstico de salidas por hora en tu estación

Un practicante nos mandó otro modelo para tu estación. Dice que se equivoca por **0.00 bicis** por hora en julio.
Antes de ponerlo en producción, ¿le creemos? Su código está abajo.

---

## El modelo del practicante (3 min)

Corran la celda tal cual. Luego hagan la pregunta de siempre para cada columna: **¿a las 7:59 ya conozco este dato
para la hora de 8 a 9?**

In [ ]:
columnas_practicante = [
    "departures_lag_24h", "departures_lag_168h", "est_hora",
    "arrivals", "net_flow",          # <- QUITA ESTA LÍNEA y vuelve a correr la celda
]
modelo_practicante = ajusta(columnas_practicante)
print(f"MAE en julio, estación {estacion}: {error_julio(modelo_practicante, columnas_practicante):.2f} bicis por hora")

Ahora **borren la línea marcada** y vuelvan a correr. Ese es el error real del modelo del practicante.

## Preguntas (3 min)

Escriban la letra en la hoja, debajo de cada pregunta (doble clic para escribir).

**1. ¿Por qué el modelo del practicante daba 0.00?**

- a) Porque la recta es muy buena
- b) Porque julio fue un mes fácil
- c) Porque usa datos de la misma hora que predice: llegadas + neto = salidas
- d) Porque tiene muchas columnas

**Su respuesta (doble clic para escribir):** ___

**2. prom_est_hora es el promedio de la estación a esa hora. ¿Con qué datos se calcula?**

- a) Con todo el año, para que sea más preciso
- b) Solo con train (octubre a junio)
- c) Solo con julio
- d) Da igual

**Su respuesta (doble clic para escribir):** ___

**3. ¿Qué diferencia hay entre prom_est_hora y est_hora?**

- a) Ninguna, son la misma columna
- b) prom_est_hora le da al modelo el promedio ya calculado; est_hora le da una columna de 0 y 1 por pareja y el modelo aprende su peso
- c) est_hora es fuga y prom_est_hora no
- d) prom_est_hora solo sirve para la estación 51

**Su respuesta (doble clic para escribir):** ___

**4. Agregas quincena y el error de tu estación casi no cambia. ¿Qué le dices a Operaciones?**

- a) Que la deje, por si acaso
- b) Que no la use: en julio no ayudó y es una columna más que mantener
- c) Que el modelo está mal
- d) Que hay fuga

**Su respuesta (doble clic para escribir):** ___